<a href="https://colab.research.google.com/github/imagra93/ML-course-labs/blob/main/deep_learning/multitask_face_lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Task Face Analysis with PyTorch Lightning

This is the companion to [`multitask_face_plain_pytorch.ipynb`](multitask_face_plain_pytorch.ipynb). Same dataset, same ResNet-50 backbone — but two things change, and it is worth being precise about which is which:

| | Plain notebook | This notebook |
|---|---|---|
| **Engineering** | Hand-written `run_epoch`, manual device moves, manual checkpointing | `LightningModule` + `Trainer` + callbacks |
| **Science** | One task (eye coordinates) despite the "multi-task" title | Three genuine tasks: **eyes**, **gender**, **age** |

The engineering change is the easy one to explain and the easy one to oversell. Lightning does not make a model better — it removes a category of silent bugs and gives you multi-GPU, checkpointing and logging for free. The *science* change is where the actual learning content of this notebook lives: once you have three heads with wildly different loss scales, you have to think about how to combine them, and that is the central topic of multi-task learning.

## Roadmap

1. **Theory**: why Lightning, anatomy of a `LightningModule`, and the multi-task loss problem
2. Config and data download
3. EDA — and a **trivial baseline** for all three tasks (do not skip this section)
4. `Dataset` with three targets
5. `LightningDataModule` — including train-split-only target statistics
6. `LightningModule` — shared trunk, three heads, weighted loss, TorchMetrics
7. `Trainer` with `EarlyStopping`, `ModelCheckpoint`, `CSVLogger`
8. Test evaluation against the baseline
9. Learning curves and visual inference
10. Reflection questions

## 1. Theory — Why Lightning?

### The problem it solves

Look at `run_epoch` in the plain notebook. It is about 25 lines, and **not one of them is about faces, eyes, or ResNets**. It is the same loop in every supervised-learning project you will ever write. Yet each line is an opportunity for a silent bug:

| Mistake | Symptom |
|---|---|
| Forget `optimizer.zero_grad()` | Gradients accumulate across batches; training quietly diverges |
| `model.train(False)` but no `torch.no_grad()` | Correct numbers, but the activation graph is built anyway → memory blows up |
| Average loss without weighting by batch size | The last (smaller) batch is over-weighted in the epoch mean |
| Move `x` to the device but forget `y` | `RuntimeError: expected all tensors on the same device` (this one at least is loud) |
| Compute `mean(sqrt(mse_per_batch))` | **Not** the epoch RMSE — see the Jensen note below |

None of these are interesting problems. They are *engineering* problems, and they are identical in every project.

### The thesis: separate research code from engineering code

You write the **what**:

- the model architecture
- the loss
- the optimizer and schedule
- the metrics

Lightning runs the **how**:

- the epoch/batch loop, in the right order
- `zero_grad` → `forward` → `backward` → `clip` → `step`
- device placement for every tensor in the batch
- `torch.no_grad()` and `model.eval()` during validation and test
- correct epoch-level aggregation of losses and metrics
- checkpointing, early stopping, logging, progress bars
- CPU / single-GPU / multi-GPU / TPU with no code change

### What Lightning is *not*

It is not a new framework and there is no lock-in. `L.LightningModule` **subclasses `torch.nn.Module`**. Any PyTorch code you can write works unchanged inside it; `self.backbone`, `state_dict()`, `.to(device)` all behave exactly as before. You can take a trained `LightningModule`, call `.state_dict()`, and load it into a plain `nn.Module` with the same layer names.

It also will not fix your science. A clean `Trainer` call that trains the wrong model on leaky data is just a tidier way of being wrong — which is exactly why section 3 of this notebook computes a baseline before training anything.

### A concrete correctness win: the Jensen gap

The plain notebook accumulates `sqrt(MSE)` per batch and averages those. Because $\sqrt{\cdot}$ is **concave**, Jensen's inequality gives

$$\frac{1}{B}\sum_b \sqrt{\text{MSE}_b} \;\le\; \sqrt{\frac{1}{B}\sum_b \text{MSE}_b}$$

so the reported "RMSE" is *systematically optimistic*. In that notebook's final epoch it printed `val_metric 0.0184` while the true epoch RMSE was $\sqrt{0.000348} = 0.0187$. A small gap here, but it is a real bias and it grows as batch-to-batch variance grows. TorchMetrics' `MeanSquaredError(squared=False)` accumulates the sum of squared errors and the count, then takes the square root **once at the end** — which is the correct quantity by construction.

## 2. Theory — Anatomy of a `LightningModule`

A `LightningModule` is an `nn.Module` with a fixed set of named hooks. You override the ones you need; Lightning calls them at the right moment.

| Hook | When it runs | What you put in it |
|---|---|---|
| `__init__` | Once | Layers, losses, metric objects, `save_hyperparameters()` |
| `forward(x)` | When *you* call it | Inference path. By convention: what you'd want at deployment |
| `training_step(batch, idx)` | Every train batch | Compute and **return** the loss |
| `validation_step(batch, idx)` | Every val batch | Compute metrics; returns nothing (already inside `no_grad` + `eval`) |
| `test_step(batch, idx)` | Every test batch, only on `trainer.test()` | Final held-out evaluation |
| `predict_step(batch, idx)` | On `trainer.predict()` | Return predictions for downstream use |
| `configure_optimizers()` | Once, at fit start | Optimizer(s) and LR scheduler(s) |

### The `training_step` contract

You **return** the loss; you never call `loss.backward()`, `optimizer.step()`, or `optimizer.zero_grad()`. Lightning does all three, in the correct order, with gradient clipping and accumulation applied if configured. This single convention is what removes the top three bugs in the table above.

### The `_shared_step` pattern

`training_step`, `validation_step` and `test_step` almost always compute the *same* forward pass and loss, differing only in what they log. Factoring that into one private `_shared_step(batch, stage)` keeps train and eval logic provably identical — a surprisingly common source of train/eval skew when they are written twice.

### Logging semantics: `on_step` vs `on_epoch`

```python
self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
```

- `on_step=True` → one logged point per batch. Noisy, but what you want to diagnose a divergence *within* an epoch.
- `on_epoch=True` → Lightning accumulates across the epoch and logs one correctly-weighted average. This is what you plot as a learning curve.

Defaults differ by hook (`training_step` defaults to step-level, `validation_step` to epoch-level), so being explicit is good practice.

### Why TorchMetrics objects instead of floats

Passing a `torchmetrics.Metric` object to `self.log` rather than a Python float buys three things:

1. **Correct aggregation.** A metric holds *state* (e.g. sum of squared errors and a count), calls `update()` per batch, and `compute()` once at epoch end. The mean of per-batch means is wrong whenever batch sizes differ; a mean of per-batch *RMSEs* is wrong always (see the Jensen note).
2. **Correct distributed reduction.** On multiple GPUs, metric states are synchronised across processes before `compute()`. Hand-rolled float accumulators silently report only rank 0's slice.
3. **Automatic lifecycle.** Lightning calls `reset()` between epochs, so metrics cannot leak from one epoch into the next.

The rule of thumb: **loss** is what you optimise (it must be differentiable); a **metric** is what you report (it does not have to be, and often shouldn't be — accuracy and MAE are not useful losses).

## 3. Theory — Multi-Task Learning

### The architecture

One shared backbone $f_\theta$ (the *trunk*), several small task-specific heads $g_{\phi_k}$:

$$\mathbf{h} = f_\theta(\mathbf{x}) \in \mathbb{R}^{2048}, \qquad \hat{y}_k = g_{\phi_k}(\mathbf{h})$$

The trunk holds ~23.5M parameters; each head here holds ~2–8K. So the tasks share essentially the entire model, and the heads are little more than linear read-outs of a representation that must serve all three.

### Why it can help

- **Inductive transfer / implicit regularisation.** The trunk cannot overfit an idiosyncrasy of one task if that would damage another. Each task acts as a data-dependent regulariser on the others.
- **Data efficiency.** Three supervision signals per image instead of one. Gender and age labels are effectively free extra gradient on the same pixels.
- **Deployment cost.** One forward pass, one set of weights, three outputs — instead of three ResNet-50s.

### Why it can hurt: negative transfer

If the tasks want *different* features, they fight. The gradients they send into the trunk can point in opposing directions:

$$\nabla_\theta \mathcal{L} = \sum_k \lambda_k \nabla_\theta \mathcal{L}_k$$

and if $\langle \nabla_\theta \mathcal{L}_1, \nabla_\theta \mathcal{L}_2 \rangle < 0$ the update helps one task at the expense of the other. This is **negative transfer**, and it is the reason MTL is not free: a multi-task model can be *worse on every task* than three single-task models. (Methods like PCGrad and GradNorm attack this directly by projecting or rescaling conflicting gradients; out of scope here, but worth knowing the names.)

### The real problem: loss scale imbalance

The total loss is a weighted sum,

$$\mathcal{L} = \sum_k \lambda_k \mathcal{L}_k$$

and the choice of $\lambda_k$ matters enormously, because the raw losses live on completely different scales. For **this** dataset, at initialisation:

| Task | Loss | Typical raw magnitude | Why |
|---|---|---|---|
| Eyes | MSE on coords in $[0,1]$ | $\approx 6\times10^{-5}$ | Coordinates barely vary — faces are aligned |
| Gender | BCE | $\approx 0.69$ | $-\ln(0.5)$, the value at chance |
| Age | MSE in years$^2$ | $\approx 73$ | $\sigma_{\text{age}} \approx 8.5$ years, so $\sigma^2 \approx 73$ |

With $\lambda_k = 1$ for all $k$, the age term is **over a million times larger** than the eye term. The gradient of the sum is, for all practical purposes, the gradient of the age loss alone. The eye task would simply not be trained.

### Two cures — and they are the same cure

**(a) Weight the losses.** Choose $\lambda_k \approx 1/\mathcal{L}_k^{\text{baseline}}$ so every term starts near 1.

**(b) Standardise the targets.** Replace $y$ with $z = (y - \mu)/\sigma$ using **training-split** statistics.

For a squared loss these are algebraically identical: scaling a target by $1/\sigma$ divides the MSE by $\sigma^2$, which is exactly a loss weight of $\lambda = 1/\sigma^2$. We use **(b)** here because it is self-documenting — after standardisation an MSE of 1.0 means "no better than predicting the mean", which is an interpretable number on every task at once — and because it makes $\lambda_k = 1$ an honest default rather than a magic constant.

The heads therefore predict **z-scores**, and the model un-standardises them back to real units (coordinates in $[0,1]$, age in years) for metrics and inference. The $\mu, \sigma$ constants are registered as **buffers**, so they are saved inside the checkpoint and a loaded model needs no external state to produce correct predictions.

> **A note on checkpoint portability.** Everything passed to `__init__` is stored in the checkpoint by `save_hyperparameters()`. Since PyTorch 2.6, `torch.load` defaults to `weights_only=True` and will refuse to unpickle arbitrary objects — including NumPy arrays. So the statistics are stored as plain Python floats and lists. Keeping hyperparameters to primitive types is the difference between a checkpoint that reloads anywhere and one that raises `UnpicklingError` on someone else's machine.

> **A note on leakage.** $\mu$ and $\sigma$ are computed on the **training split only**, inside the `DataModule`'s `setup()`. Computing them over the full CSV would leak information about validation and test targets into training — a small leak here, a fatal one in other settings. The habit is what matters.

### An aside: learned loss weights

You do not have to pick $\lambda_k$ by hand. Kendall et al. (2018) treat each task's weight as a learned *homoscedastic uncertainty* $\sigma_k$, minimising

$$\mathcal{L} = \sum_k \frac{1}{2\sigma_k^2}\mathcal{L}_k + \log \sigma_k$$

where $\log \sigma_k$ is a free parameter. The first term down-weights noisy tasks; the $\log\sigma_k$ term stops the model from cheating by sending all $\sigma_k \to \infty$. It is about fifteen lines to implement and is a good extension exercise (see the reflection questions).

## 4. Setup

The install cell is idempotent: it only fetches what is actually missing, so it is a no-op on a machine that already has the dependencies, and does the right thing on a fresh Colab runtime.

In [ ]:
# Install only what is missing (no-op locally, installs on a fresh Colab runtime)
import importlib.util, subprocess, sys

for _pkg in ["lightning", "torchmetrics", "gdown"]:
    if importlib.util.find_spec(_pkg) is None:
        print(f"Installing {_pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
print("Dependencies ready.")

In [ ]:
import math
import os
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torchmetrics import MeanAbsoluteError, MeanSquaredError
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC

print("torch     ", torch.__version__)
print("lightning ", L.__version__)
print("cuda       ", torch.cuda.is_available())

### Configuration as a dataclass

Every tunable in one frozen place. This is not cosmetic: `asdict(cfg)` is trivially loggable, so a run's full configuration ends up next to its metrics, and reproducing a result six months later does not depend on remembering which cell you edited.

In [ ]:
@dataclass
class Config:
    # data
    data_csv: str = "data/data.csv"
    images_dir: str = "data/images"
    img_size: int = 224
    val_frac: float = 0.15
    test_frac: float = 0.15

    # optimisation
    batch_size: int = 32
    max_epochs: int = 12
    lr: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.3
    grad_clip: float = 1.0

    # multi-task loss weights (targets are standardised, so 1.0 is a fair default)
    w_eyes: float = 1.0
    w_gender: float = 1.0
    w_age: float = 1.0

    # engineering
    num_workers: int = 8
    patience: int = 5
    seed: int = 42
    log_dir: str = "logs_lightning"


cfg = Config()
L.seed_everything(cfg.seed, workers=True)
print(asdict(cfg))

## 5. Data Download

Guarded so re-running the notebook does not re-download 35 MB.

In [ ]:
import zipfile

import gdown

if not Path(cfg.data_csv).exists():
    file_id = "1c6Gv3LB2mi3_7_gzN1q6e6echCk3sinF"
    out = "downloaded_file.zip"
    gdown.download(f"https://drive.google.com/uc?id={file_id}", out, quiet=False)
    with zipfile.ZipFile(out, "r") as zf:
        zf.extractall("data")
    os.remove(out)
    print("Download and extraction completed.")
else:
    print(f"{cfg.data_csv} already present — skipping download.")

## 6. EDA — and why this dataset is harder to read than it looks

In [ ]:
df = pd.read_csv(cfg.data_csv)
print(f"rows: {len(df)}   columns: {list(df.columns)}")
print(f"missing values: {df.isna().sum().sum()}")
display(df.head())
display(df[["left_eye_x", "left_eye_y", "right_eye_x", "right_eye_y", "age"]].describe())

In [ ]:
# Are the images a uniform size? This matters more than it usually does — see below.
sizes = {Image.open(Path(cfg.images_dir) / f).size for f in df["im_name"].sample(300, random_state=0)}
print("distinct image sizes in a 300-image sample:", sizes)

W, H = next(iter(sizes))
eyes_norm = np.stack([
    df["left_eye_x"] / W, df["left_eye_y"] / H,
    df["right_eye_x"] / W, df["right_eye_y"] / H,
], axis=1)

print("\nnormalised eye coordinates")
print("  mean:", eyes_norm.mean(0).round(4))
print("  std :", eyes_norm.std(0).round(4))
print(f"\n  -> std in pixels: {(eyes_norm.std(0) * np.array([W, H, W, H])).round(2)}")

**This is the single most important observation in the notebook.** Every image is the same size and the faces are *aligned* — this is a CelebA-style crop where the eyes have already been registered to near-identical positions. The standard deviation of the eye coordinates is around **1–2 pixels**.

That means the eye-localisation task is close to degenerate: a model that ignores the image entirely and always outputs the mean eye position will score very well. Any loss number you see for this task is therefore meaningless *in isolation*. The next section makes that concrete.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df["gender"].value_counts().plot(kind="bar", ax=axes[0], title="Gender balance", rot=0)
axes[0].set_ylabel("count")

df["age"].plot(kind="hist", bins=30, ax=axes[1], title="Age distribution")
axes[1].set_xlabel("age (years)")

axes[2].scatter(eyes_norm[:, 0], eyes_norm[:, 1], s=2, alpha=0.2, label="left eye")
axes[2].scatter(eyes_norm[:, 2], eyes_norm[:, 3], s=2, alpha=0.2, label="right eye")
axes[2].set_xlim(0, 1); axes[2].set_ylim(1, 0)
axes[2].set_title("Eye positions (normalised, full frame)")
axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
# A few samples with their ground-truth eyes marked
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for ax, (_, row) in zip(axes, df.head(5).iterrows()):
    img = Image.open(Path(cfg.images_dir) / row["im_name"]).convert("RGB")
    d = ImageDraw.Draw(img)
    for x, y in [(row["left_eye_x"], row["left_eye_y"]), (row["right_eye_x"], row["right_eye_y"])]:
        d.ellipse([x - 4, y - 4, x + 4, y + 4], outline="lime", width=2)
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{row['gender']}, {row['age']}y", fontsize=10)
plt.tight_layout(); plt.show()

## 7. Baselines — do this *before* you train anything

A loss of 0.0003 sounds excellent. It is meaningless until you know what a model that has learned **nothing** would score.

For each task there is a trivial predictor that uses no input features at all:

| Task | Trivial predictor | Metric |
|---|---|---|
| Eyes | always the training-mean coordinate | RMSE (normalised units) |
| Gender | always the majority class | accuracy |
| Age | always the training-mean age | MAE (years) |

These are the numbers your model has to **beat**. If it does not, it has not learned to use the image — and a low loss is measuring the variance of the dataset, not the quality of the model. Reporting a model without its baseline is one of the most common ways to fool yourself in applied ML.

In [ ]:
rng = np.random.default_rng(cfg.seed)
_perm = rng.permutation(len(df))
_n_test = int(cfg.test_frac * len(df))
_n_val = int(cfg.val_frac * len(df))
_idx_test = _perm[:_n_test]
_idx_train = _perm[_n_test + _n_val:]

_age = df["age"].to_numpy(dtype=float)
_male = (df["gender"] == "male").to_numpy()

BASELINE = {
    "eyes_rmse": float(np.sqrt(((eyes_norm[_idx_test] - eyes_norm[_idx_train].mean(0)) ** 2).mean())),
    "gender_acc": float(max(_male[_idx_test].mean(), 1 - _male[_idx_test].mean())),
    "age_mae": float(np.abs(_age[_idx_test] - _age[_idx_train].mean()).mean()),
}

print("Trivial baselines on the held-out test split")
print(f"  eyes  RMSE : {BASELINE['eyes_rmse']:.4f}  (= {BASELINE['eyes_rmse'] * cfg.img_size:.2f} px at {cfg.img_size}px)")
print(f"  gender acc : {BASELINE['gender_acc']:.4f}")
print(f"  age   MAE  : {BASELINE['age_mae']:.2f} years")
print("\nRaw loss magnitudes at initialisation (the scale-imbalance problem, measured):")
print(f"  eyes   MSE (coords in [0,1]) : {((eyes_norm[_idx_test] - eyes_norm[_idx_train].mean(0)) ** 2).mean():.2e}")
print(f"  gender BCE (at chance)       : {math.log(2):.2e}")
print(f"  age    MSE (years^2)         : {_age.var():.2e}")
print(f"\n  -> ratio age:eyes = {_age.var() / ((eyes_norm[_idx_test] - eyes_norm[_idx_train].mean(0)) ** 2).mean():,.0f} : 1")

## 8. Dataset

### Why almost no augmentation

The plain notebook uses none; here we add **photometric** augmentation only (`ColorJitter`), and that restriction is deliberate:

> **Geometric augmentation breaks keypoint labels.** `RandomHorizontalFlip` mirrors the pixels but *not* the coordinates — and worse, after a flip the left eye is on the right. You would have to transform the targets alongside the image and swap the two eye labels. `RandomCrop` and `RandomRotation` have the same problem.

Colour, brightness and contrast changes leave every coordinate exactly where it was, so they are safe for all three tasks. This is a general rule for keypoint and detection problems: augment geometry only if your pipeline transforms the labels too (libraries like Albumentations exist precisely for this).

In [ ]:
def build_transforms(img_size: int):
    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size), antialias=True),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size), antialias=True),
        transforms.ToTensor(),
        norm,
    ])
    return train_tf, eval_tf

In [ ]:
class FaceMultiTaskDataset(Dataset):
    """Returns one sample with all three targets in *real* units.

    eyes   : (4,) float32, normalised to [0,1] by the original image size
    gender : () float32, 1.0 = male, 0.0 = female  (BCEWithLogits wants a float)
    age    : () float32, in years
    """

    def __init__(self, frame: pd.DataFrame, root_dir, transform=None):
        self.data = frame.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform

        expected = ["im_name", "gender", "left_eye_x", "left_eye_y",
                    "right_eye_x", "right_eye_y", "age"]
        missing = [c for c in expected if c not in self.data.columns]
        if missing:
            raise ValueError(f"DataFrame is missing columns: {missing}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path = self.root_dir / str(row["im_name"])
        image = Image.open(path).convert("RGB")
        w, h = image.width, image.height

        eyes = torch.tensor([
            float(row["left_eye_x"]) / w, float(row["left_eye_y"]) / h,
            float(row["right_eye_x"]) / w, float(row["right_eye_y"]) / h,
        ], dtype=torch.float32).clamp_(0.0, 1.0)

        return {
            "image": self.transform(image) if self.transform else image,
            "eyes": eyes,
            "gender": torch.tensor(float(row["gender"] == "male"), dtype=torch.float32),
            "age": torch.tensor(float(row["age"]), dtype=torch.float32),
            "path": str(path),
        }

## 9. `LightningDataModule`

A `DataModule` bundles *everything* about the data — splits, transforms, dataloaders, and any statistics derived from the training set — behind five named methods. The payoff is that the model becomes data-agnostic and the data becomes model-agnostic: you can swap either side without touching the other.

| Method | Called | Purpose |
|---|---|---|
| `prepare_data()` | Once, **on one process only** | Downloads, one-off writes to disk. Never set state here |
| `setup(stage)` | On **every** process | Splits, `Dataset` construction, statistics |
| `train/val/test_dataloader()` | Per epoch | Return the `DataLoader` |

The one-process/all-process split matters: in distributed training `prepare_data` runs once (so eight processes do not race to write the same zip file) while `setup` runs everywhere (so every process builds its own identical split).

**Where the target statistics live.** `setup()` computes `mean`/`std` of the eye coordinates and of age **from the training indices only**, and exposes them as `self.target_stats`. This is the anti-leakage discipline from section 3 turned into code — and it is also why the `DataModule` is the right home for it: the statistics are a property of the *data split*, not of the model.

Two loader details worth knowing:

- `persistent_workers=True` keeps the worker processes alive between epochs. Without it, Python re-forks `num_workers` processes every single epoch, which on a short epoch can cost more than the epoch itself.
- `pin_memory=True` stages batches in page-locked memory so the host→GPU copy can be asynchronous.

In [ ]:
class FaceDataModule(L.LightningDataModule):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.target_stats = None

    def prepare_data(self):
        # Runs on a single process. The download cell above already did this;
        # the check keeps the DataModule self-contained if you reuse it elsewhere.
        if not Path(self.cfg.data_csv).exists():
            raise FileNotFoundError(f"{self.cfg.data_csv} not found — run the download cell.")

    def setup(self, stage=None):
        frame = pd.read_csv(self.cfg.data_csv)
        n = len(frame)

        # Deterministic split, independent of `stage` so every call agrees
        g = np.random.default_rng(self.cfg.seed)
        perm = g.permutation(n)
        n_test = int(self.cfg.test_frac * n)
        n_val = int(self.cfg.val_frac * n)
        idx_test, idx_val, idx_train = perm[:n_test], perm[n_test:n_test + n_val], perm[n_test + n_val:]

        train_tf, eval_tf = build_transforms(self.cfg.img_size)
        self.train_ds = FaceMultiTaskDataset(frame.iloc[idx_train], self.cfg.images_dir, train_tf)
        self.val_ds = FaceMultiTaskDataset(frame.iloc[idx_val], self.cfg.images_dir, eval_tf)
        self.test_ds = FaceMultiTaskDataset(frame.iloc[idx_test], self.cfg.images_dir, eval_tf)

        # ---- statistics from the TRAINING SPLIT ONLY (no leakage) ----
        tr = frame.iloc[idx_train]
        sizes = {Image.open(Path(self.cfg.images_dir) / f).size for f in tr["im_name"].head(50)}
        w, h = next(iter(sizes))
        eyes_tr = np.stack([
            tr["left_eye_x"] / w, tr["left_eye_y"] / h,
            tr["right_eye_x"] / w, tr["right_eye_y"] / h,
        ], axis=1)
        n_male = int((tr["gender"] == "male").sum())

        # Plain Python types only — these end up in the checkpoint via
        # save_hyperparameters(), and torch>=2.6 loads with weights_only=True,
        # which refuses to unpickle numpy arrays.
        self.target_stats = {
            "eyes_mu": eyes_tr.mean(0).tolist(),
            "eyes_sd": eyes_tr.std(0).tolist(),
            "age_mu": float(tr["age"].mean()),
            "age_sd": float(tr["age"].std()),
            # BCEWithLogits pos_weight = n_negative / n_positive, counteracts class imbalance
            "gender_pos_weight": float((len(tr) - n_male) / max(n_male, 1)),
        }

    def _loader(self, ds, shuffle):
        return DataLoader(
            ds,
            batch_size=self.cfg.batch_size,
            shuffle=shuffle,
            num_workers=self.cfg.num_workers,
            pin_memory=True,
            persistent_workers=self.cfg.num_workers > 0,
        )

    def train_dataloader(self):
        return self._loader(self.train_ds, True)

    def val_dataloader(self):
        return self._loader(self.val_ds, False)

    def test_dataloader(self):
        return self._loader(self.test_ds, False)

In [ ]:
dm = FaceDataModule(cfg)
dm.prepare_data()
dm.setup()   # called explicitly here because the model needs the target statistics

print(f"train / val / test : {len(dm.train_ds)} / {len(dm.val_ds)} / {len(dm.test_ds)}")
for k, v in dm.target_stats.items():
    print(f"  {k:18s} {np.round(v, 4)}")

## 10. The `LightningModule`

### Architecture

```
image (B, 3, 224, 224)
        |
  ResNet-50 trunk (ImageNet weights, fc removed)      ~23.5M params, SHARED
        |
    h (B, 2048)  ->  Dropout(0.3)
        |
   +----+--------------+--------------+
   |                   |              |
 eyes_head         gender_head     age_head
 Linear(2048,4)   Linear(2048,1)  Linear(2048,1)
   |                   |              |
 z-scores (4)       logit          z-score
```

### Two output spaces, and why

The heads emit **z-scores** (and a raw logit). That is the space the loss lives in — after the standardisation of section 3, every task's MSE starts near 1.0 and the weights $\lambda_k = 1$ are fair.

But z-scores are useless to a caller. So `forward()` — the *inference* path — un-standardises them and returns real units: coordinates in $[0,1]$, age in years, a gender probability. Metrics are computed in that same real space, which is what makes them directly comparable to the baselines from section 7.

The $\mu,\sigma$ constants are `register_buffer`s, not plain attributes. Buffers are part of `state_dict()` but are not parameters: they move with `.to(device)`, they are saved into the checkpoint, and they are not updated by the optimizer. A checkpoint therefore carries everything needed to produce correct predictions — no side-channel of pickled scalers to keep in sync.

### Losses

| Task | Loss | Note |
|---|---|---|
| Eyes | `MSELoss` on z-scores | |
| Age | `MSELoss` on z-score | |
| Gender | `BCEWithLogitsLoss` | Takes the **logit**, not a probability |

`BCEWithLogitsLoss` rather than `Sigmoid` + `BCELoss`: it fuses the two and uses the log-sum-exp trick, which is numerically stable when the logit saturates. `sigmoid` + `log` separately will produce `inf` gradients at confident predictions. The `pos_weight` argument multiplies the positive class's contribution by $n_{\text{neg}}/n_{\text{pos}}$ to offset the 58/42 imbalance.

### `configure_optimizers` with a scheduler

Returning the dict form lets Lightning drive a `ReduceLROnPlateau`, which needs to be told *which* metric to watch and how often to step. `monitor="val/loss"` is a string reference to a logged key — if you log nothing by that name, Lightning raises at the first epoch boundary rather than failing silently.

In [ ]:
class MultiTaskFaceNet(L.LightningModule):
    def __init__(self, target_stats: dict, lr: float, weight_decay: float, dropout: float,
                 w_eyes: float, w_gender: float, w_age: float):
        super().__init__()
        # Stores every __init__ arg in self.hparams AND in the checkpoint, so
        # load_from_checkpoint() can rebuild the model with no arguments.
        self.save_hyperparameters()

        # ---- shared trunk ----
        trunk = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat_dim = trunk.fc.in_features
        trunk.fc = nn.Identity()
        self.trunk = trunk
        self.dropout = nn.Dropout(dropout)

        # ---- task heads (predict z-scores / logits) ----
        self.eyes_head = nn.Linear(feat_dim, 4)
        self.gender_head = nn.Linear(feat_dim, 1)
        self.age_head = nn.Linear(feat_dim, 1)

        # ---- target standardisation constants: buffers travel with the checkpoint ----
        self.register_buffer("eyes_mu", torch.tensor(target_stats["eyes_mu"], dtype=torch.float32))
        self.register_buffer("eyes_sd", torch.tensor(target_stats["eyes_sd"], dtype=torch.float32))
        self.register_buffer("age_mu", torch.tensor(float(target_stats["age_mu"])))
        self.register_buffer("age_sd", torch.tensor(float(target_stats["age_sd"])))

        # ---- losses ----
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(float(target_stats["gender_pos_weight"]))
        )

        # ---- metrics: one independent copy per stage, in REAL units ----
        # NB the "_metrics" suffix is not cosmetic: a ModuleDict key of "train"
        # would collide with nn.Module.train() and raise at construction time.
        self.metrics = nn.ModuleDict({
            f"{stage}_metrics": nn.ModuleDict({
                "eyes_rmse": MeanSquaredError(squared=False),
                "age_mae": MeanAbsoluteError(),
                "gender_acc": BinaryAccuracy(),
                "gender_auroc": BinaryAUROC(),
            })
            for stage in ("train", "val", "test")
        })

    # ---------------- inference path: real units ----------------
    def forward(self, x):
        z = self._heads(x)
        return {
            "eyes": z["eyes"] * self.eyes_sd + self.eyes_mu,   # [0,1] coordinates
            "age": z["age"] * self.age_sd + self.age_mu,       # years
            "gender_prob": torch.sigmoid(z["gender_logit"]),   # P(male)
        }

    # ---------------- training path: z-score space ----------------
    def _heads(self, x):
        h = self.dropout(self.trunk(x))
        return {
            "eyes": self.eyes_head(h),
            "gender_logit": self.gender_head(h).squeeze(1),
            "age": self.age_head(h).squeeze(1),
        }

    def _shared_step(self, batch, stage: str):
        z = self._heads(batch["image"])

        # standardise the targets into the space the heads predict in
        t_eyes = (batch["eyes"] - self.eyes_mu) / self.eyes_sd
        t_age = (batch["age"] - self.age_mu) / self.age_sd

        loss_eyes = self.mse(z["eyes"], t_eyes)
        loss_age = self.mse(z["age"], t_age)
        loss_gender = self.bce(z["gender_logit"], batch["gender"])

        loss = (self.hparams.w_eyes * loss_eyes
                + self.hparams.w_gender * loss_gender
                + self.hparams.w_age * loss_age)

        # back to real units for the metrics
        p_eyes = z["eyes"] * self.eyes_sd + self.eyes_mu
        p_age = z["age"] * self.age_sd + self.age_mu
        p_gender = torch.sigmoid(z["gender_logit"])
        target_gender = batch["gender"].int()

        m = self.metrics[f"{stage}_metrics"]
        m["eyes_rmse"].update(p_eyes, batch["eyes"])
        m["age_mae"].update(p_age, batch["age"])
        m["gender_acc"].update(p_gender, target_gender)
        m["gender_auroc"].update(p_gender, target_gender)

        on_bar = stage == "val"
        self.log(f"{stage}/loss", loss, on_step=False, on_epoch=True, prog_bar=on_bar)
        self.log(f"{stage}/loss_eyes", loss_eyes, on_step=False, on_epoch=True)
        self.log(f"{stage}/loss_gender", loss_gender, on_step=False, on_epoch=True)
        self.log(f"{stage}/loss_age", loss_age, on_step=False, on_epoch=True)
        for name, metric in m.items():
            self.log(f"{stage}/{name}", metric, on_step=False, on_epoch=True,
                     prog_bar=on_bar and name == "eyes_rmse")
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")     # return the loss; Lightning does the rest

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        out = self(batch["image"])
        return {k: v.cpu() for k, v in out.items()} | {"path": batch["path"]}

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val/loss", "interval": "epoch"},
        }

## 11. Training

### Callbacks

A callback is behaviour that hooks into the loop without being part of the model — precisely the engineering/science boundary again.

| Callback | What it does |
|---|---|
| `ModelCheckpoint` | Saves the **best** weights by a monitored metric, not just the last |
| `EarlyStopping` | Halts when the monitored metric stops improving for `patience` epochs |
| `LearningRateMonitor` | Logs the LR each epoch, so a scheduler's effect is visible in the curves |

The best/last distinction matters: the final epoch is rarely the best epoch. Evaluating the last weights on test is a quiet way to report a worse number than you earned — or, if you early-stopped on test, a better one than you deserve.

### `Trainer`

`gradient_clip_val=1.0` becomes a flag rather than a line in your loop. `deterministic` and `accelerator="auto"` handle reproducibility and hardware. The same call runs on CPU, one GPU, or eight.

In [ ]:
model = MultiTaskFaceNet(
    target_stats=dm.target_stats,
    lr=cfg.lr, weight_decay=cfg.weight_decay, dropout=cfg.dropout,
    w_eyes=cfg.w_eyes, w_gender=cfg.w_gender, w_age=cfg.w_age,
)

n_trunk = sum(p.numel() for p in model.trunk.parameters())
n_heads = sum(p.numel() for n, p in model.named_parameters() if "head" in n)
print(f"trunk (shared): {n_trunk:,} params")
print(f"heads (3 tasks): {n_heads:,} params  ->  {n_heads / n_trunk:.2%} of the trunk")

In [ ]:
ckpt_cb = ModelCheckpoint(monitor="val/loss", mode="min", save_top_k=1,
                          save_last=True, filename="best-epoch{epoch:02d}",
                          auto_insert_metric_name=False)
early_cb = EarlyStopping(monitor="val/loss", mode="min", patience=cfg.patience)
lr_cb = LearningRateMonitor(logging_interval="epoch")
logger = CSVLogger(save_dir=cfg.log_dir, name="multitask")

trainer = L.Trainer(
    max_epochs=cfg.max_epochs,
    accelerator="auto",
    devices=1,
    logger=logger,
    callbacks=[ckpt_cb, early_cb, lr_cb],
    gradient_clip_val=cfg.grad_clip,
    log_every_n_steps=25,
    enable_progress_bar=True,
)

trainer.fit(model, datamodule=dm)
print("\nBest checkpoint:", ckpt_cb.best_model_path)
print("Best val/loss  :", float(ckpt_cb.best_model_score))

## 12. Test evaluation

`ckpt_path="best"` tells Lightning to reload the best checkpoint before evaluating — not the final-epoch weights sitting in memory. Test runs **once**, at the end. Every time you look at the test set and then change something, it becomes a little more like a validation set.

In [ ]:
test_results = trainer.test(model, datamodule=dm, ckpt_path="best")[0]
test_results

### Did it actually learn anything?

Now the baselines from section 7 earn their keep.

In [ ]:
comparison = pd.DataFrame({
    "metric": ["eyes RMSE (norm)", "gender accuracy", "age MAE (years)"],
    "trivial baseline": [BASELINE["eyes_rmse"], BASELINE["gender_acc"], BASELINE["age_mae"]],
    "multi-task model": [test_results["test/eyes_rmse"], test_results["test/gender_acc"],
                         test_results["test/age_mae"]],
    "better when": ["lower", "higher", "lower"],
})
comparison["beats baseline"] = [
    comparison.loc[0, "multi-task model"] < comparison.loc[0, "trivial baseline"],
    comparison.loc[1, "multi-task model"] > comparison.loc[1, "trivial baseline"],
    comparison.loc[2, "multi-task model"] < comparison.loc[2, "trivial baseline"],
]
display(comparison.round(4))
print(f"\ngender AUROC: {test_results['test/gender_auroc']:.4f}  (0.5 = chance)")

## 13. Learning curves

`CSVLogger` writes one row per logging event to `metrics.csv`. Train and validation metrics are logged at different steps, so the file is sparse — most columns are `NaN` in most rows. Grouping by epoch and taking the last non-null value per column reassembles the per-epoch series.

In [ ]:
metrics = pd.read_csv(Path(logger.log_dir) / "metrics.csv")
per_epoch = metrics.groupby("epoch").last(numeric_only=True)

panels = [
    ("loss", "total weighted loss"),
    ("loss_eyes", "eyes loss (z-space)"),
    ("loss_gender", "gender BCE"),
    ("loss_age", "age loss (z-space)"),
    ("eyes_rmse", "eyes RMSE (normalised)"),
    ("age_mae", "age MAE (years)"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (key, title) in zip(axes.ravel(), panels):
    for split, style in [("train", "-o"), ("val", "-s")]:
        col = f"{split}/{key}"
        if col in per_epoch and per_epoch[col].notna().any():
            s = per_epoch[col].dropna()
            ax.plot(s.index, s.values, style, ms=4, label=split)
    if key in ("eyes_rmse", "age_mae"):
        ax.axhline(BASELINE[key], color="crimson", ls="--", lw=1.2, label="trivial baseline")
    ax.set_title(title); ax.set_xlabel("epoch")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for split, style in [("train", "-o"), ("val", "-s")]:
    col = f"{split}/gender_acc"
    if col in per_epoch and per_epoch[col].notna().any():
        s = per_epoch[col].dropna()
        ax.plot(s.index, s.values, style, ms=4, label=split)
ax.axhline(BASELINE["gender_acc"], color="crimson", ls="--", lw=1.2, label="majority class")
ax.set_title("gender accuracy"); ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 14. Visual inference

`trainer.predict()` runs `predict_step` over a dataloader with `eval()` and `no_grad()` already applied, on the right device. We reload the best checkpoint with `load_from_checkpoint` — which needs no constructor arguments, because `save_hyperparameters()` stored them all, and no external scaler, because $\mu,\sigma$ are buffers.

Green = ground truth, red = prediction.

In [ ]:
best_model = MultiTaskFaceNet.load_from_checkpoint(ckpt_cb.best_model_path)
best_model.eval()

batch = next(iter(dm.test_dataloader()))
with torch.no_grad():
    out = best_model(batch["image"].to(best_model.device))

n_show = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
for i, ax in enumerate(axes.ravel()[:n_show]):
    img = Image.open(batch["path"][i]).convert("RGB")
    d = ImageDraw.Draw(img)
    w, h = img.size

    gt = batch["eyes"][i].tolist()
    pr = out["eyes"][i].cpu().tolist()
    for coords, colour in [(gt, "lime"), (pr, "red")]:
        for (x, y) in [(coords[0] * w, coords[1] * h), (coords[2] * w, coords[3] * h)]:
            d.rectangle([x - 5, y - 5, x + 5, y + 5], outline=colour, width=2)

    p_male = out["gender_prob"][i].item()
    ax.imshow(img); ax.axis("off")
    ax.set_title(
        f"age {out['age'][i].item():.0f} (true {batch['age'][i].item():.0f})\n"
        f"{'male' if p_male > 0.5 else 'female'} p={max(p_male, 1 - p_male):.2f} "
        f"(true {'male' if batch['gender'][i] > 0.5 else 'female'})",
        fontsize=9,
    )
plt.tight_layout(); plt.show()

## 15. Plain PyTorch vs Lightning — what actually disappeared

| Concern | Plain notebook | This notebook |
|---|---|---|
| Epoch loop | `run_epoch`, ~25 lines | `trainer.fit()` |
| `zero_grad` / `backward` / `step` | Manual, order-sensitive | Implicit in the `training_step` contract |
| Device placement | `.to(DEVICE)` on each tensor | Automatic for every tensor in the batch |
| `no_grad` during validation | Manual `with torch.no_grad():` | Automatic in `validation_step` |
| Epoch metric aggregation | Manual, batch-size weighted by hand | TorchMetrics, correct by construction |
| Gradient clipping | `clip_grad_norm_` call | `gradient_clip_val=1.0` flag |
| Best-model checkpointing | Hand-rolled `if val_loss < best` | `ModelCheckpoint(monitor=...)` |
| Early stopping | `PATIENCE` defined but **never implemented** | `EarlyStopping(patience=...)` |
| LR scheduling | Manual `scheduler.step(val_loss)` | Declared in `configure_optimizers` |
| Metric logging | `history` dict, appended by hand | `self.log` → `CSVLogger`/TensorBoard/W&B |
| Multi-GPU | Rewrite with DDP | `devices=8` |
| Reproducibility of config | Module-level globals | `Config` dataclass + `save_hyperparameters()` |

The line count is roughly a wash. What changes is that the remaining lines are all about *your problem*, and the deleted ones were all about *the loop* — where the bugs were. Note one entry in particular: the plain notebook defines `PATIENCE = 5` and never uses it. That is exactly the class of omission that a declarative callback makes impossible to forget.

## Where to go next

- **Discriminative learning rates.** Give the pre-trained trunk a lower LR than the freshly-initialised heads (two param groups in `configure_optimizers`). Usually worth a point or two on transfer-learning tasks.
- **Freeze then unfreeze.** Train the heads only for an epoch or two (`requires_grad=False` on the trunk) before fine-tuning end to end, so random heads do not send garbage gradients into good features.
- **Mixed precision.** `precision="16-mixed"` in the `Trainer` — roughly 2× faster on modern GPUs, one argument.
- **Uncertainty weighting.** Implement the Kendall et al. loss from section 3 and compare the learned $\lambda_k$ against our standardisation.

## 16. Reflection Questions

1. **The baseline result.** Look at the comparison table in section 12. If the model does *not* beat the trivial baseline on eye localisation, is that a bug, or a property of the dataset? What would you change to find out?

2. **Standardisation vs loss weights.** We standardised the targets and set all $\lambda_k = 1$. Show algebraically that for a squared loss, dividing the target by $\sigma$ is exactly equivalent to a loss weight of $1/\sigma^2$. Does the same equivalence hold for the gender BCE term?

3. **Leakage.** `setup()` computes $\mu,\sigma$ from the training indices only. Suppose you computed them over the whole CSV instead. Exactly what leaks, and would you be able to detect it from the validation curves?

4. **`prepare_data` vs `setup`.** Why does Lightning insist that `prepare_data` runs on one process while `setup` runs on all of them? What breaks if you assign `self.train_ds` inside `prepare_data` and train on 4 GPUs?

5. **Buffers vs parameters.** `eyes_mu` is a buffer, not a `nn.Parameter`. What three things would change if you had used `nn.Parameter` instead?

6. **Augmentation.** Why is `ColorJitter` safe here but `RandomHorizontalFlip` is not? Write the label transformation that *would* make horizontal flipping correct for this dataset.

7. **Negative transfer.** How would you test whether the three tasks are helping or hurting each other, using only this notebook's machinery?

## Answers

**1. The baseline result.**
Almost certainly a property of the dataset, not a bug. The images are aligned CelebA-style crops, so the eye coordinates have a standard deviation of 1–2 pixels — there is very little signal to predict. A constant predictor is a *strong* baseline precisely because the target has almost no variance. To confirm it is not a bug: check that the model beats the baseline on *gender* (which has real signal). If gender works and eyes do not, the pipeline is fine and the eye task is simply near-degenerate on this data. The fix is a better dataset — unaligned, varied-pose faces — not a bigger model.

**2. Standardisation vs loss weights.**
With $z = y/\sigma$ and $\hat z = \hat y/\sigma$: $\text{MSE}(\hat z, z) = \mathbb{E}[(\hat y/\sigma - y/\sigma)^2] = \frac{1}{\sigma^2}\mathbb{E}[(\hat y - y)^2] = \frac{1}{\sigma^2}\text{MSE}(\hat y, y)$. So the two are identical up to the constant $1/\sigma^2$, which is exactly a loss weight. **Not** for BCE: cross-entropy takes a probability, which is already bounded in $[0,1]$ and cannot be rescaled the same way — there is no "standard deviation of a Bernoulli target" to divide out. Its scale is fixed by the entropy of the label distribution, which is why the gender term needs no rescaling to start at $O(1)$.

**3. Leakage.**
The mean and standard deviation of the validation and test *targets* would leak into the training objective. The model would be trained in a coordinate system defined partly by data it is supposed to be evaluated on, so validation and test scores become mildly optimistic. You would **not** see it in the curves — that is exactly what makes this class of bug dangerous. It is undetectable from inside the experiment and only shows up as a gap when the model meets genuinely new data.

**4. `prepare_data` vs `setup`.**
`prepare_data` is for side effects on shared resources — downloading, extracting, writing a cache. Running it on all 4 processes means 4 concurrent writes to the same path: a corrupted zip, at best a wasted 4× download. `setup` builds *in-memory* state that each process needs its own copy of. If you assign `self.train_ds` inside `prepare_data`, only rank 0 gets it; ranks 1–3 hit `AttributeError`, because Lightning does not (and cannot cheaply) broadcast arbitrary Python objects between processes.

**5. Buffers vs parameters.**
(i) `nn.Parameter` has `requires_grad=True`, so $\mu,\sigma$ would receive gradients and **drift during training** — the target space would move under the model. (ii) They would be picked up by `self.parameters()` and so handed to the optimizer, including weight decay, which would pull them toward zero. (iii) They would appear in parameter counts and in `named_parameters()`. Both buffers and parameters are in `state_dict()` and both follow `.to(device)` — that part is unchanged.

**6. Augmentation.**
`ColorJitter` alters pixel *intensities*; every coordinate keeps its position, so all three labels stay valid. A horizontal flip moves the pixels but not the labels. The correct label transform is a mirror **and** a swap, because after flipping, the anatomical left eye appears on the right of the image:
```python
eyes_flipped = torch.tensor([1 - eyes[2], eyes[3], 1 - eyes[0], eyes[1]])
```
Mirroring $x \mapsto 1-x$ without the swap is the subtle version of the bug: the coordinates land in the right places, but the two eyes are labelled backwards, and the model learns a contradiction.

**7. Negative transfer.**
Train the same architecture with one head at a time (`w_gender=0, w_age=0`, then the other combinations — the weights are already `Config` fields, so no code changes are needed) and compare each task's test metric against the three-task run. If the single-task model is better on a task, the other tasks are hurting it. A cheap intermediate diagnostic: log the cosine similarity between $\nabla_\theta \mathcal{L}_k$ for each pair of tasks on the last trunk block; persistently negative values are direct evidence of conflicting gradients.